# Notebook 01c — Bilingual Lexicon Baselines (SentiLexM + MELex)

**Project:** Benchmarking Modern Multilingual Transformer Models for Malay-English Code-Switched Sentiment Analysis  
**Authors:** Joshua Joenathan Thomas (25141571) | Amit Kumar Gupta (25109952)

## Purpose — Addressing the Critic's "Strawman" Objection (v2 Lexicon Baseline)

The original study (Notebook 01b) evaluated only English-centric lexicon tools: VADER, TextBlob, and pysentimiento. A critic correctly identified this as a strawman comparison — these tools were never designed for Malay or code-switched text, so their poor performance does not rigorously demonstrate that *lexicon-based approaches* are inadequate for this domain.

This notebook implements **two bilingual Malay-English lexicons** as proper zero-shot baselines:

| Lexicon | Entries | Coverage | Domain | Role |
|---------|---------|----------|--------|------|
| **SentiLexM** | 26,004 | 2,477 English + 23,527 Malay (incl. slang) | General social media | **Primary** |
| **MELex** | 6,132 | Malay + English (no language tags) | Property-sector Twitter | Secondary (direct critic citation) |

### Why SentiLexM is the domain-appropriate primary choice

The critic cited MELex as the appropriate bilingual baseline. However, MELex (Husna et al.) was constructed from **property-sector Malaysian Twitter** — housing discourse, real-estate promotions, developer complaints. Its sentiment vocabulary is tuned for that domain and its reported accuracy comes from in-domain evaluation, not general social media.

**SentiLexM** (Azlan et al., MIT license) is built on AFINN-111 with Malay translations and explicit slang coverage, making it domain-appropriate for general social media code-switching. It is the correct tool for MESocSentiment.

MELex is implemented here solely as a direct response to the critic's specific citation, with clear labelling of its domain limitation.

### V2 evaluation sets
Both the **v1 test set** (2,000 samples, original study comparability) and the **v2 clean test set** (1,700 samples, gold-split for corrected experiments) are evaluated.

### Expected outcome
Even domain-appropriate bilingual lexicons will leave a gap to supervised approaches — the gap measures the structural limitations of static word-score lookup, not a deficiency of bilingual resource choice.

In [1]:
import sys
sys.path.insert(0, r'D:\Doc\Programming in AI\Project\src')  # absolute path -- works regardless of Jupyter launch dir

import re, os, json, requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from config import TEST_CSV, CLEAN_TEST_CSV, RESULTS_DIR, SEED, LABEL2ID, ID2LABEL

BILINGUAL_DIR = RESULTS_DIR / 'bilingual_lexicon'
BILINGUAL_DIR.mkdir(parents=True, exist_ok=True)

# Negation tokens covering both English and Malay
NEGATION_TOKENS = {
    'tidak', 'tak', 'bukan', 'tiada', 'xde', 'xnak', 'xleh',
    'jangan', 'never', 'not', "n't", 'no', 'neither', 'nor',
    'belum', 'tanpa',
}
NEGATION_WINDOW = 3  # tokens to look back for negation

LABEL_ORDER = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']

print('Config loaded.')
print(f'RESULTS_DIR   : {RESULTS_DIR}')
print(f'BILINGUAL_DIR : {BILINGUAL_DIR}')
print(f'TEST_CSV      : {TEST_CSV}')
print(f'CLEAN_TEST_CSV: {CLEAN_TEST_CSV}')

Config loaded.
RESULTS_DIR   : D:\Doc\Programming in AI\Project\results
BILINGUAL_DIR : D:\Doc\Programming in AI\Project\results\bilingual_lexicon
TEST_CSV      : D:\Doc\Programming in AI\Project\Datasets\MESocSentiment-main\MESocSentiment Corpus\Test Set.csv
CLEAN_TEST_CSV: D:\Doc\Programming in AI\Project\results\test_clean_test_split.csv


In [2]:
# ── Preprocessing helper ────────────────────────────────────────────────────
def preprocess(text: str) -> str:
    """Lowercase, strip URLs. Retain hashtags and special characters
    (consistent with Notebook 01b preprocessing)."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    return text.strip()


# ── V1 gold test set (2,000 samples) ────────────────────────────────────────
df_v1_raw = pd.read_csv(TEST_CSV)
df_v1_raw.columns = ['text', 'label_auto', 'label_manual']
df_v1 = df_v1_raw[['text', 'label_manual']].copy()
df_v1['label'] = df_v1['label_manual'].str.upper().str.strip()
df_v1['text_clean'] = df_v1['text'].apply(preprocess)
print(f'V1 test set: {len(df_v1):,} samples')
print(f'V1 class distribution: {dict(df_v1["label"].value_counts())}')


# ── V2 clean test set (1,700 samples) ───────────────────────────────────────
if not Path(CLEAN_TEST_CSV).exists():
    print()
    print('WARNING: CLEAN_TEST_CSV not found at:')
    print(f'  {CLEAN_TEST_CSV}')
    print('The v2 (1,700-sample) evaluation will be SKIPPED.')
    print('Run Notebook 01 first to generate the gold-split CSV.')
    df_v2 = None
else:
    df_v2 = pd.read_csv(CLEAN_TEST_CSV)
    # Normalise column names — CLEAN_TEST_CSV may use 'text'/'label' directly
    if 'label' not in df_v2.columns and 'label_manual' in df_v2.columns:
        df_v2['label'] = df_v2['label_manual'].str.upper().str.strip()
    elif 'label' not in df_v2.columns:
        candidate_cols = [c for c in df_v2.columns
                          if 'manual' in c.lower() or 'sentiment' in c.lower()]
        if candidate_cols:
            df_v2['label'] = df_v2[candidate_cols[0]].str.upper().str.strip()
        else:
            raise ValueError(
                f'Cannot identify label column in CLEAN_TEST_CSV. '
                f'Columns: {list(df_v2.columns)}'
            )
    if 'text_clean' not in df_v2.columns:
        text_col = 'text' if 'text' in df_v2.columns else df_v2.columns[0]
        df_v2['text_clean'] = df_v2[text_col].apply(preprocess)
    print(f'V2 clean test set: {len(df_v2):,} samples')
    print(f'V2 class distribution: {dict(df_v2["label"].value_counts())}')

V1 test set: 2,000 samples
V1 class distribution: {'NEUTRAL': np.int64(1386), 'NEGATIVE': np.int64(388), 'POSITIVE': np.int64(226)}
V2 clean test set: 1,700 samples
V2 class distribution: {'NEUTRAL': np.int64(1178), 'NEGATIVE': np.int64(330), 'POSITIVE': np.int64(192)}


In [3]:
# ── Download and parse SentiLexM ────────────────────────────────────────────
# Source: https://github.com/AsyrafAzlan/SentiLexM
# Format: tab-separated, word<TAB>score (no header).
# Score range: -5 to +5 (AFINN scale).
# 26,004 entries: 2,477 English + 23,527 Malay (including slang). All lowercase.

SENTILEXM_URL = 'https://raw.githubusercontent.com/AsyrafAzlan/SentiLexM/main/SentiLexM.txt'

print('Downloading SentiLexM...')
try:
    response = requests.get(SENTILEXM_URL, timeout=15)
    response.raise_for_status()
    raw_text = response.text
    print(f'Downloaded {len(raw_text):,} bytes.')
except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f'Failed to download SentiLexM from:\n  {SENTILEXM_URL}\n'
        f'Error: {e}\n'
        'Check your internet connection, or download the file manually and '
        'adapt the parsing code below to load from a local path.'
    )

sentilexm: dict = {}
parse_errors = 0

for line in raw_text.splitlines():
    line = line.strip()
    if not line:
        continue
    parts = line.split('\t')
    if len(parts) != 2:
        parse_errors += 1
        continue
    word, score_str = parts
    try:
        sentilexm[word.lower()] = float(score_str)
    except ValueError:
        parse_errors += 1

# Statistics
scores = np.array(list(sentilexm.values()))
pos_count = int((scores > 0).sum())
neg_count = int((scores < 0).sum())
neu_count = int((scores == 0).sum())

print(f'\n=== SentiLexM Statistics ===')
print(f'Total entries : {len(sentilexm):,}')
print(f'Positive (>0) : {pos_count:,}')
print(f'Negative (<0) : {neg_count:,}')
print(f'Neutral  (=0) : {neu_count:,}')
print(f'Parse errors  : {parse_errors}')
print(f'Score range   : [{scores.min():.1f}, {scores.max():.1f}]')
print(f'Score mean    : {scores.mean():.3f}  std: {scores.std():.3f}')

# Score distribution plot
fig, ax = plt.subplots(figsize=(8, 4))
unique_scores, counts = np.unique(scores, return_counts=True)
ax.bar(unique_scores, counts, color='steelblue', edgecolor='white', width=0.6)
ax.set_xlabel('Sentiment Score (AFINN scale)')
ax.set_ylabel('Number of Entries')
ax.set_title('SentiLexM — Score Distribution (26,004 entries)')
ax.set_xticks(sorted(unique_scores))
plt.tight_layout()
plt.savefig(BILINGUAL_DIR / 'sentilexm_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sentilexm_score_distribution.png')

Downloaded 365,585 bytes.

=== SentiLexM Statistics ===
Total entries : 24,765
Positive (>0) : 8,082
Negative (<0) : 16,682
Neutral  (=0) : 1
Parse errors  : 0
Score range   : [-5.0, 5.0]
Score mean    : -0.833  std: 2.100
Saved: sentilexm_score_distribution.png


C:\Users\Josh\AppData\Local\Temp\ipykernel_8384\967160207.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Store SentiLexM to SQLite
# Satisfies spec requirement: datasets stored in DB prior to processing.
import sqlite3
from config import DB_PATH

df_sentilexm = pd.DataFrame(
    list(sentilexm.items()),
    columns=["word", "score"]
)
conn = sqlite3.connect(DB_PATH)
df_sentilexm.to_sql("sentilexm", conn, if_exists="replace", index=False)
conn.close()
print(f"Stored {len(df_sentilexm):,} SentiLexM entries -> SQLite table sentilexm ({DB_PATH.name})")

Stored 24,765 SentiLexM entries -> SQLite table sentilexm (mesocsentiment.db)


In [5]:
# ── Download and parse MELex ─────────────────────────────────────────────────
# Source: https://github.com/nhusna84/MELex
# Format: colon-delimited, word:score (no header). Score range: -3 to +3.
# 6,132 entries. Domain: property-sector Twitter. No language tags.
#
# NOTE: MELex is implemented as a SECONDARY reference to directly respond to
# the critic's specific citation. It was constructed from Malaysian property-sector
# Twitter (real-estate discussions, housing complaints, developer promotions) and
# its sentiment vocabulary is biased toward that domain. For general social media
# code-switching, SentiLexM (Cell 3) is the domain-appropriate choice.

MELEX_URL = 'https://raw.githubusercontent.com/nhusna84/MELex/master/MELex_E4.txt'

print('Downloading MELex...')
try:
    response = requests.get(MELEX_URL, timeout=15)
    response.raise_for_status()
    raw_text_melex = response.text
    print(f'Downloaded {len(raw_text_melex):,} bytes.')
except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f'Failed to download MELex from:\n  {MELEX_URL}\n'
        f'Error: {e}\n'
        'Check your internet connection.'
    )

melex: dict = {}
melex_parse_errors = 0

for line in raw_text_melex.splitlines():
    line = line.strip()
    if not line:
        continue
    # Use rsplit with maxsplit=1 to correctly handle words that contain colons
    parts = line.rsplit(':', 1)
    if len(parts) != 2:
        melex_parse_errors += 1
        continue
    word, score_str = parts
    try:
        melex[word.strip().lower()] = float(score_str.strip())
    except ValueError:
        melex_parse_errors += 1

# Statistics
melex_scores = np.array(list(melex.values()))
melex_pos = int((melex_scores > 0).sum())
melex_neg = int((melex_scores < 0).sum())
melex_neu = int((melex_scores == 0).sum())

print(f'\n=== MELex Statistics ===')
print(f'Total entries : {len(melex):,}')
print(f'Positive (>0) : {melex_pos:,}')
print(f'Negative (<0) : {melex_neg:,}')
print(f'Neutral  (=0) : {melex_neu:,}')
print(f'Parse errors  : {melex_parse_errors}')
print(f'Score range   : [{melex_scores.min():.1f}, {melex_scores.max():.1f}]')
print(f'Score mean    : {melex_scores.mean():.3f}  std: {melex_scores.std():.3f}')

# Vocabulary overlap with SentiLexM
overlap = set(sentilexm.keys()) & set(melex.keys())
print(f'\nVocabulary overlap with SentiLexM: {len(overlap):,} shared tokens')


Downloaded 74,121 bytes.

=== MELex Statistics ===
Total entries : 5,759
Positive (>0) : 3,339
Negative (<0) : 2,420
Neutral  (=0) : 0
Parse errors  : 0
Score range   : [-3.0, 3.0]
Score mean    : 0.180  std: 1.105

Vocabulary overlap with SentiLexM: 1,695 shared tokens


In [6]:
# Store MELex to SQLite
df_melex = pd.DataFrame(
    list(melex.items()),
    columns=["word", "score"]
)
conn = sqlite3.connect(DB_PATH)
df_melex.to_sql("melex", conn, if_exists="replace", index=False)
conn.close()
print(f"Stored {len(df_melex):,} MELex entries -> SQLite table melex ({DB_PATH.name})")

Stored 5,759 MELex entries -> SQLite table melex (mesocsentiment.db)


In [7]:
def classify_with_lexicon(
    text,
    lexicon: dict,
    negation_window: int = NEGATION_WINDOW
) -> str:
    """
    Classify sentiment of `text` using a word-score lexicon.

    Algorithm:
    1. Tokenize: text.lower().split()
    2. For each token found in the lexicon:
       - Look back at the previous `negation_window` tokens.
       - If any negation token is found in that window, negate the score (* -1).
       - Add the (possibly negated) score to the running total.
    3. Return 'POSITIVE' if total > 0,
              'NEGATIVE' if total < 0,
              'NEUTRAL'  if total == 0.

    Edge cases handled:
    - Non-string input (including None): coerced to string.
    - Empty text or no lexicon matches: returns 'NEUTRAL'.
    - All matched scores sum to exactly zero: returns 'NEUTRAL'.
    """
    if not isinstance(text, str):
        text = str(text) if text is not None else ''

    tokens = text.lower().split()

    if not tokens:
        return 'NEUTRAL'

    total_score = 0.0

    for i, token in enumerate(tokens):
        if token not in lexicon:
            continue
        score = lexicon[token]
        # Check the negation_window tokens immediately before this token
        window_start = max(0, i - negation_window)
        preceding = tokens[window_start:i]
        if any(t in NEGATION_TOKENS for t in preceding):
            score *= -1
        total_score += score

    if total_score > 0:
        return 'POSITIVE'
    elif total_score < 0:
        return 'NEGATIVE'
    else:
        return 'NEUTRAL'


# ── Quick sanity checks ──────────────────────────────────────────────────────
test_cases = [
    ('i love this',          sentilexm, 'POSITIVE'),
    ('i hate this',          sentilexm, 'NEGATIVE'),
    ('i do not love this',   sentilexm, 'NEGATIVE'),
    ('tak suka langsung',    sentilexm, 'NEGATIVE'),   # 'tak' negates 'suka' (like)
    ('',                     sentilexm, 'NEUTRAL'),
    (None,                   sentilexm, 'NEUTRAL'),
]

print('=== Sanity checks ===')
all_pass = True
for text, lex, expected in test_cases:
    result = classify_with_lexicon(text, lex)
    status = 'PASS' if result == expected else 'FAIL'
    if status == 'FAIL':
        all_pass = False
    print(f'  [{status}] {repr(str(text)):40s} expected={expected}  got={result}')
print(f'\nAll sanity checks passed: {all_pass}')

=== Sanity checks ===
  [PASS] 'i love this'                            expected=POSITIVE  got=POSITIVE
  [PASS] 'i hate this'                            expected=NEGATIVE  got=NEGATIVE
  [PASS] 'i do not love this'                     expected=NEGATIVE  got=NEGATIVE
  [PASS] 'tak suka langsung'                      expected=NEGATIVE  got=NEGATIVE
  [PASS] ''                                       expected=NEUTRAL  got=NEUTRAL
  [PASS] 'None'                                   expected=NEUTRAL  got=NEUTRAL

All sanity checks passed: True


In [8]:
# ── Shared evaluation helper ─────────────────────────────────────────────────

def evaluate_lexicon(df, lexicon, lexicon_name, split_name, label_order=LABEL_ORDER):
    """Run predictions and return (result_dict, pd.Series of predictions)."""
    preds = df['text_clean'].apply(lambda t: classify_with_lexicon(t, lexicon))
    labels = df['label']

    macro_f1 = f1_score(
        labels, preds, average='macro', labels=label_order, zero_division=0
    )
    report_dict = classification_report(
        labels, preds, labels=label_order, target_names=label_order,
        output_dict=True, zero_division=0
    )

    print(f'=== {lexicon_name} — {split_name} ===')
    print(f'Macro-F1: {macro_f1:.4f}')
    print(classification_report(
        labels, preds, labels=label_order, target_names=label_order, zero_division=0
    ))
    print(f'Prediction distribution: {dict(preds.value_counts())}')
    print()

    result = {
        'model': lexicon_name,
        'split': split_name,
        'type': 'bilingual_lexicon',
        'macro_f1': round(macro_f1, 4),
        'per_class': {
            cls: {
                'precision': round(report_dict[cls]['precision'], 4),
                'recall':    round(report_dict[cls]['recall'], 4),
                'f1':        round(report_dict[cls]['f1-score'], 4),
                'support':   int(report_dict[cls]['support']),
            }
            for cls in label_order
        },
        'n_samples': len(df),
    }
    return result, preds


# ── SentiLexM — V1 (2,000 samples) ──────────────────────────────────────────
slm_v1_result, slm_v1_preds = evaluate_lexicon(
    df_v1, sentilexm, 'SentiLexM', 'v1_gold_2000'
)

# ── SentiLexM — V2 (1,700 samples) — only if available ──────────────────────
slm_v2_result = None
slm_v2_preds = None
if df_v2 is not None:
    slm_v2_result, slm_v2_preds = evaluate_lexicon(
        df_v2, sentilexm, 'SentiLexM', 'v2_clean_1700'
    )

# ── Save results ─────────────────────────────────────────────────────────────
sentilexm_all_results = {
    'lexicon': 'SentiLexM',
    'source': 'https://github.com/AsyrafAzlan/SentiLexM',
    'n_entries': len(sentilexm),
    'negation_window': NEGATION_WINDOW,
    'v1_gold_2000': slm_v1_result,
    'v2_clean_1700': slm_v2_result,
}
out_path = BILINGUAL_DIR / 'sentilexm_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(sentilexm_all_results, f, indent=2)
print(f'Saved: {out_path}')

# ── Confusion matrix — V1 ────────────────────────────────────────────────────
cm = confusion_matrix(df_v1['label'], slm_v1_preds, labels=LABEL_ORDER)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=LABEL_ORDER).plot(ax=ax, cmap='Blues')
ax.set_title(
    f'SentiLexM — V1 Gold Test Set (Macro-F1 = {slm_v1_result["macro_f1"]:.4f})',
    fontsize=11
)
plt.tight_layout()
plt.savefig(BILINGUAL_DIR / 'sentilexm_cm_v1.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Confusion matrix — V2 (if available) ────────────────────────────────────
if df_v2 is not None and slm_v2_preds is not None:
    cm2 = confusion_matrix(df_v2['label'], slm_v2_preds, labels=LABEL_ORDER)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm2, display_labels=LABEL_ORDER).plot(ax=ax, cmap='Blues')
    ax.set_title(
        f'SentiLexM — V2 Clean Test Set (Macro-F1 = {slm_v2_result["macro_f1"]:.4f})',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(BILINGUAL_DIR / 'sentilexm_cm_v2.png', dpi=150, bbox_inches='tight')
    plt.show()

=== SentiLexM — v1_gold_2000 ===
Macro-F1: 0.4049
              precision    recall  f1-score   support

    NEGATIVE       0.31      0.39      0.35       388
     NEUTRAL       0.80      0.42      0.55      1386
    POSITIVE       0.20      0.72      0.32       226

    accuracy                           0.45      2000
   macro avg       0.44      0.51      0.40      2000
weighted avg       0.64      0.45      0.48      2000

Prediction distribution: {'POSITIVE': np.int64(791), 'NEUTRAL': np.int64(727), 'NEGATIVE': np.int64(482)}

=== SentiLexM — v2_clean_1700 ===
Macro-F1: 0.4053
              precision    recall  f1-score   support

    NEGATIVE       0.32      0.40      0.36       330
     NEUTRAL       0.80      0.42      0.55      1178
    POSITIVE       0.20      0.70      0.31       192

    accuracy                           0.45      1700
   macro avg       0.44      0.51      0.41      1700
weighted avg       0.64      0.45      0.48      1700

Prediction distribution: {'POS

C:\Users\Josh\AppData\Local\Temp\ipykernel_8384\1144822072.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Josh\AppData\Local\Temp\ipykernel_8384\1144822072.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ── Evaluate MELex ────────────────────────────────────────────────────────────
#
# DOMAIN LIMITATION NOTE:
# MELex was built from property-sector Malaysian Twitter data. Its lexicon entries
# reflect vocabulary from real-estate discussions (e.g. 'beli' [buy], 'mahal'
# [expensive], 'rosak' [broken/faulty]) rather than the general-purpose social media
# expressions in MESocSentiment. This domain mismatch is expected to reduce its
# effectiveness here relative to SentiLexM. Results are included for completeness
# and to directly respond to the critic's citation of MELex as the missing baseline.

# V1 (2,000 samples)
melex_v1_result, melex_v1_preds = evaluate_lexicon(
    df_v1, melex, 'MELex', 'v1_gold_2000'
)

# V2 (1,700 samples) — only if available
melex_v2_result = None
melex_v2_preds = None
if df_v2 is not None:
    melex_v2_result, melex_v2_preds = evaluate_lexicon(
        df_v2, melex, 'MELex', 'v2_clean_1700'
    )

# Save results
melex_all_results = {
    'lexicon': 'MELex',
    'source': 'https://github.com/nhusna84/MELex',
    'n_entries': len(melex),
    'domain_note': (
        'Built from property-sector Malaysian Twitter. '
        'Domain mismatch with general social media expected.'
    ),
    'negation_window': NEGATION_WINDOW,
    'v1_gold_2000': melex_v1_result,
    'v2_clean_1700': melex_v2_result,
}
out_path = BILINGUAL_DIR / 'melex_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(melex_all_results, f, indent=2)
print(f'Saved: {out_path}')

# Confusion matrix — V1
cm = confusion_matrix(df_v1['label'], melex_v1_preds, labels=LABEL_ORDER)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=LABEL_ORDER).plot(ax=ax, cmap='Oranges')
ax.set_title(
    f'MELex — V1 Gold Test Set (Macro-F1 = {melex_v1_result["macro_f1"]:.4f})',
    fontsize=11
)
plt.tight_layout()
plt.savefig(BILINGUAL_DIR / 'melex_cm_v1.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion matrix — V2 (if available)
if df_v2 is not None and melex_v2_preds is not None:
    cm2 = confusion_matrix(df_v2['label'], melex_v2_preds, labels=LABEL_ORDER)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm2, display_labels=LABEL_ORDER).plot(ax=ax, cmap='Oranges')
    ax.set_title(
        f'MELex — V2 Clean Test Set (Macro-F1 = {melex_v2_result["macro_f1"]:.4f})',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(BILINGUAL_DIR / 'melex_cm_v2.png', dpi=150, bbox_inches='tight')
    plt.show()

=== MELex — v1_gold_2000 ===
Macro-F1: 0.2804
              precision    recall  f1-score   support

    NEGATIVE       0.24      0.21      0.22       388
     NEUTRAL       0.73      0.28      0.40      1386
    POSITIVE       0.13      0.66      0.22       226

    accuracy                           0.31      2000
   macro avg       0.37      0.38      0.28      2000
weighted avg       0.57      0.31      0.35      2000

Prediction distribution: {'POSITIVE': np.int64(1138), 'NEUTRAL': np.int64(521), 'NEGATIVE': np.int64(341)}

=== MELex — v2_clean_1700 ===
Macro-F1: 0.2811
              precision    recall  f1-score   support

    NEGATIVE       0.23      0.20      0.21       330
     NEUTRAL       0.73      0.29      0.41      1178
    POSITIVE       0.13      0.66      0.22       192

    accuracy                           0.31      1700
   macro avg       0.37      0.38      0.28      1700
weighted avg       0.57      0.31      0.35      1700

Prediction distribution: {'POSITIVE':

C:\Users\Josh\AppData\Local\Temp\ipykernel_8384\1059917344.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Josh\AppData\Local\Temp\ipykernel_8384\1059917344.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ── Comparison table: all lexicons ───────────────────────────────────────────
#
# V1 results from Notebook 01b (hardcoded as paper values; overridden if JSON
# exists from a prior run of that notebook).

V1_PAPER_RESULTS = [
    {
        'model': 'VADER',
        'macro_f1': 0.3827,
        'type': 'english_lexicon',
        'per_class': {
            'NEGATIVE': {'f1': 0.1014},
            'NEUTRAL':  {'f1': 0.7263},
            'POSITIVE': {'f1': 0.3205},
        },
    },
    {
        'model': 'TextBlob',
        'macro_f1': 0.3747,
        'type': 'english_lexicon',
        'per_class': {
            'NEGATIVE': {'f1': 0.0922},
            'NEUTRAL':  {'f1': 0.7551},
            'POSITIVE': {'f1': 0.2767},
        },
    },
    {
        'model': 'pysentimiento (en)',
        'macro_f1': 0.4778,
        'type': 'english_lexicon',
        'per_class': {
            'NEGATIVE': {'f1': 0.1330},
            'NEUTRAL':  {'f1': 0.8324},
            'POSITIVE': {'f1': 0.4680},
        },
    },
]

existing_results_path = RESULTS_DIR / 'lexicon' / 'all_lexicon_results.json'
if existing_results_path.exists():
    with open(existing_results_path, 'r', encoding='utf-8') as f:
        v1_results = json.load(f)
    print(f'Loaded existing v1 lexicon results from {existing_results_path}')
else:
    v1_results = V1_PAPER_RESULTS
    print('Using hardcoded v1 paper values (results/lexicon/all_lexicon_results.json not found).')

# Build comparison rows (all on V1 / 2,000-sample set for consistent comparison)
comparison_rows = []
for res in v1_results:
    comparison_rows.append({
        'Model':         res['model'],
        'Type':          res.get('type', 'lexicon'),
        'Macro-F1 (v1)': res['macro_f1'],
        'NEG-F1':        res.get('per_class', {}).get('NEGATIVE', {}).get('f1', float('nan')),
        'NEU-F1':        res.get('per_class', {}).get('NEUTRAL',  {}).get('f1', float('nan')),
        'POS-F1':        res.get('per_class', {}).get('POSITIVE', {}).get('f1', float('nan')),
    })

# Add new bilingual results (V1)
for result, display_label in [
    (slm_v1_result,   'SentiLexM (bilingual, primary)'),
    (melex_v1_result, 'MELex (bilingual, property-sector)'),
]:
    comparison_rows.append({
        'Model':         display_label,
        'Type':          'bilingual_lexicon',
        'Macro-F1 (v1)': result['macro_f1'],
        'NEG-F1':        result['per_class']['NEGATIVE']['f1'],
        'NEU-F1':        result['per_class']['NEUTRAL']['f1'],
        'POS-F1':        result['per_class']['POSITIVE']['f1'],
    })

df_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values('Macro-F1 (v1)', ascending=False)
    .reset_index(drop=True)
)

print('\n=== Lexicon Comparison (V1 Gold Test Set, n=2,000) ===')
print(df_comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# ── Bar chart ────────────────────────────────────────────────────────────────
colors = [
    '#e07b39' if 'bilingual' in row['Type'] else '#4c78a8'
    for _, row in df_comparison.iterrows()
]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    df_comparison['Model'],
    df_comparison['Macro-F1 (v1)'],
    color=colors,
    edgecolor='white'
)
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.,
        height + 0.005,
        f'{height:.4f}',
        ha='center', va='bottom', fontsize=9
    )

ax.set_ylim(0, max(df_comparison['Macro-F1 (v1)']) * 1.18)
ax.set_ylabel('Macro-F1')
ax.set_title('Lexicon Baselines — Macro-F1 on V1 Gold Test Set (n=2,000)')
ax.tick_params(axis='x', rotation=20)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4c78a8', label='English-only lexicon (Notebook 01b)'),
    Patch(facecolor='#e07b39', label='Bilingual lexicon (this notebook)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
chart_path = BILINGUAL_DIR / 'lexicon_comparison.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {chart_path}')

# Save comparison JSON
comparison_json_path = BILINGUAL_DIR / 'lexicon_comparison.json'
with open(comparison_json_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_rows, f, indent=2)
print(f'Saved: {comparison_json_path}')

Loaded existing v1 lexicon results from D:\Doc\Programming in AI\Project\results\lexicon\all_lexicon_results.json

=== Lexicon Comparison (V1 Gold Test Set, n=2,000) ===
                             Model              Type  Macro-F1 (v1)  NEG-F1  NEU-F1  POS-F1
                pysentimiento (en)           lexicon         0.4778  0.1330  0.8324  0.4680
    SentiLexM (bilingual, primary) bilingual_lexicon         0.4049  0.3471  0.5490  0.3186
                             VADER           lexicon         0.3827  0.1014  0.7263  0.3205
                          TextBlob           lexicon         0.3747  0.0922  0.7551  0.2767
MELex (bilingual, property-sector) bilingual_lexicon         0.2804  0.2222  0.4006  0.2185
Saved: D:\Doc\Programming in AI\Project\results\bilingual_lexicon\lexicon_comparison.png
Saved: D:\Doc\Programming in AI\Project\results\bilingual_lexicon\lexicon_comparison.json


C:\Users\Josh\AppData\Local\Temp\ipykernel_8384\1100336084.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ── Save combined results for Notebook 03 analysis ───────────────────────────
#
# Produces all_lexicon_results_v2.json — consumed by downstream analysis notebooks.
# Contains all lexicons with both v1 and (where available) v2 evaluation results.

BILINGUAL_DIR.mkdir(parents=True, exist_ok=True)

combined = {
    'description': (
        'All lexicon baseline results (v2 — including bilingual resources). '
        'SentiLexM is the primary bilingual baseline; MELex is secondary '
        '(property-sector domain). '
        'English-only results (VADER, TextBlob, pysentimiento) are from Notebook 01b.'
    ),
    'english_only_v1': v1_results,
    'bilingual': {
        'SentiLexM': {
            'role': 'primary',
            'domain': 'general_social_media',
            'n_entries': len(sentilexm),
            'v1_gold_2000': slm_v1_result,
            'v2_clean_1700': slm_v2_result,
        },
        'MELex': {
            'role': 'secondary_critic_citation',
            'domain': 'property_sector_twitter',
            'n_entries': len(melex),
            'v1_gold_2000': melex_v1_result,
            'v2_clean_1700': melex_v2_result,
        },
    },
}

combined_path = BILINGUAL_DIR / 'all_lexicon_results_v2.json'
with open(combined_path, 'w', encoding='utf-8') as f:
    json.dump(combined, f, indent=2)
print(f'Saved combined results: {combined_path}')

# ── Summary table across both test sets ─────────────────────────────────────
print()
print('=== SentiLexM vs MELex — Both Evaluation Sets ===')
header = (
    f'{"Model":<38} {"Split":<18} '
    f'{"Macro-F1":>9} {"NEG-F1":>8} {"NEU-F1":>8} {"POS-F1":>8}'
)
print(header)
print('-' * len(header))

for res in [slm_v1_result, slm_v2_result, melex_v1_result, melex_v2_result]:
    if res is None:
        continue
    print(
        f'{res["model"]:<38} {res["split"]:<18}',
        f'{res["macro_f1"]:>9.4f}',
        f'{res["per_class"]["NEGATIVE"]["f1"]:>8.4f}',
        f'{res["per_class"]["NEUTRAL"]["f1"]:>8.4f}',
        f'{res["per_class"]["POSITIVE"]["f1"]:>8.4f}',
    )

print()
print('Note: NEUTRAL dominates the test set (~69% of samples).')
print('Macro-F1 weights all three classes equally, penalising poor NEG/POS performance.')
print()
print('Files written under:', BILINGUAL_DIR)

Saved combined results: D:\Doc\Programming in AI\Project\results\bilingual_lexicon\all_lexicon_results_v2.json

=== SentiLexM vs MELex — Both Evaluation Sets ===
Model                                  Split               Macro-F1   NEG-F1   NEU-F1   POS-F1
----------------------------------------------------------------------------------------------
SentiLexM                              v1_gold_2000          0.4049   0.3471   0.5490   0.3186
SentiLexM                              v2_clean_1700         0.4053   0.3550   0.5490   0.3118
MELex                                  v1_gold_2000          0.2804   0.2222   0.4006   0.2185
MELex                                  v2_clean_1700         0.2811   0.2131   0.4117   0.2186

Note: NEUTRAL dominates the test set (~69% of samples).
Macro-F1 weights all three classes equally, penalising poor NEG/POS performance.

Files written under: D:\Doc\Programming in AI\Project\results\bilingual_lexicon


## Discussion — Gravity Check

### What SentiLexM achieves

SentiLexM is the most linguistically appropriate zero-shot baseline for this corpus: it explicitly covers Malay slang, code-switching vocabulary, and is built on the AFINN scale rather than property-sector annotations. Its results reflect the realistic upper bound for a static, unsupervised bilingual lexicon on general social media.

### Why a gap to supervised approaches remains

Even with bilingual coverage, lexicon-based classifiers have structural limitations that prevent them from closing the gap to SVM or transformer baselines:

1. **No morphological analysis.** Malay affixation (prefix *ber-*, *me-*, *ke-*; suffix *-kan*, *-an*, *-lah*) changes word meaning and polarity. SentiLexM stores root forms; inflected variants go unmatched. For example, *'memalukan'* (shameful) may not match *'malu'* (shame/shy).

2. **No phonetic spelling normalisation.** Social media Malay uses extensive phonetic abbreviation: *'xde'* (tiada, nothing), *'nk'* (nak, want), *'xleh'* (tak boleh, cannot). While SentiLexM includes some slang entries, creative misspellings are unpredictable and coverage is not exhaustive.

3. **Fixed negation window.** The window-based heuristic (3 tokens) does not handle long-range negation, discontinuous constituents, or double negation patterns common in Malay (*'bukan tidak suka'* = 'not disliking').

4. **No contextual disambiguation.** The word *'sakit'* (sick/hurt) is negative in *'hati sakit'* (hurt feelings) but sarcastic/neutral in *'siapa suruh sakit'*. Static lexicon look-up cannot resolve this.

5. **Class imbalance sensitivity.** The test set is heavily skewed toward NEUTRAL (~69%). A lexicon defaulting to NEUTRAL on unknown tokens inflates neutral accuracy at the cost of NEGATIVE and POSITIVE F1, as reflected in the low per-class F1 scores for minority classes.

### MELex domain limitation

MELex was constructed from Malaysian property-sector Twitter (housing, real estate, developer promotions). Its sentiment vocabulary is tuned for that domain — words like *'beli'* (buy) or *'mahal'* (expensive) carry domain-specific sentiment weights that may not generalise. The lower Macro-F1 relative to SentiLexM on this general social media corpus reflects that domain mismatch. MELex is implemented here solely to directly address the critic's specific citation; it is not the recommended primary bilingual baseline for this task.

### Addressing the critic's objection

The critic argued that using VADER, TextBlob, and pysentimiento as lexicon baselines was a strawman — these tools were never designed for Malay or code-switched text. This notebook directly addresses that objection by implementing two genuine bilingual Malay-English resources. The measured Macro-F1 of SentiLexM now provides a fair, domain-appropriate upper bound for unsupervised lexicon approaches.

Any remaining gap between SentiLexM and supervised models (SVM, XLM-R, mDeBERTa, XLM-T) is attributable to the structural limitations listed above — not to the choice of a weak English-only baseline. That gap therefore legitimately motivates fine-tuned multilingual transformers that learn contextual, morphologically-aware representations directly from annotated code-switched data.